In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import os
from gpt import MiniGPT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


In [2]:
import os
import torch
import sys
from bpe_tokenizer import BPETokenizer
sys.modules['__main__'].BPETokenizer = BPETokenizer

train_file = 'wikitext-2-train.txt' if os.path.exists('wikitext-2-train.txt') else 'train.txt'
train_text = open(train_file, 'r', encoding='utf-8').read()
val_text   = open('wikitext-2-valid.txt', 'r', encoding='utf-8').read()
print(f"Train file size: {len(train_text):,} chars")

vocab_data = torch.load('vocab.bin', map_location='cpu', weights_only=False)
bpe = vocab_data['bpe']

if not hasattr(bpe, 'merge_patterns') or not bpe.merge_patterns:
    bpe.build_vocab()

vocab_size = len(bpe.stoi)
print(f"BPE Vocab size: {vocab_size}")
TRAIN_CHARS = 3_000_000
VAL_CHARS   =   300_000

print(f"Encoding {TRAIN_CHARS:,} train chars and {VAL_CHARS:,} val chars...")
import time
t0 = time.time()
encoded_train = bpe.encode_ids(train_text[:TRAIN_CHARS])
encoded_val   = bpe.encode_ids(val_text[:VAL_CHARS])
print(f"Encoded in {time.time()-t0:.1f}s  |  train tokens: {len(encoded_train):,}  |  val tokens: {len(encoded_val):,}")

train_data = torch.tensor(encoded_train, dtype=torch.long)
val_data   = torch.tensor(encoded_val,   dtype=torch.long)

Loading text...
Train file size: 7,329,181 chars
Loading pre-trained BPE Tokenizer from vocab.bin...
BPE Vocab size: 7108
Encoding 3,000,000 train chars and 300,000 val chars...
Encoded in 1616.0s  |  train tokens: 724,428  |  val tokens: 71,256


In [3]:
batch_size     = 64
context_length = 256
max_iters      = 10000
eval_interval  = 500
eval_iters     = 100   # batches per eval estimate
warmup_iters   = 500   # cosine LR warmup steps
learning_rate  = 3e-4
d_model        = 384
num_heads      = 12    # 384/12 = 32 dims per head
num_layers     = 6
dropout        = 0.1
grad_clip      = 1.0   # prevent exploding gradients
patience       = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length]   for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * (0.1 + 0.9 * coeff)

In [4]:
model = MiniGPT(vocab_size, d_model, num_heads, num_layers, context_length, dropout).to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

print("Starting training...")
best_val_loss = float('inf')
no_improvement_count = 0

for iter in range(max_iters):
    # Update learning rate
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Evaluate periodically
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        val_ppl = math.exp(losses['val'])
        print(f"step {iter:5d} | lr {lr:.2e} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | val ppl {val_ppl:.2f}")

        # Save best checkpoint
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), 'minigpt_weights.pth')
            no_improvement_count = 0
        else:
            # Skip checking early stopping for step 0 (initial evaluation check)
            if iter > 0:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    print(f"Early stopping triggered! Validation loss did not improve for {patience} consecutive evaluation intervals.")
                    break

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f} | Best val ppl: {math.exp(best_val_loss):.2f}")
print("Best checkpoint saved to minigpt_weights.pth")

Model parameters: 13,475,328
Starting training...
step     0 | lr 0.00e+00 | train loss 8.9404 | val loss 8.9489 | val ppl 7699.29
step   500 | lr 3.00e-04 | train loss 5.2637 | val loss 5.4606 | val ppl 235.25
step  1000 | lr 2.98e-04 | train loss 4.1109 | val loss 4.8277 | val ppl 124.92
step  1500 | lr 2.93e-04 | train loss 3.3541 | val loss 4.7197 | val ppl 112.13
step  2000 | lr 2.84e-04 | train loss 2.7031 | val loss 4.7981 | val ppl 121.29
step  2500 | lr 2.72e-04 | train loss 2.1694 | val loss 4.9492 | val ppl 141.06
step  3000 | lr 2.56e-04 | train loss 1.7489 | val loss 5.1490 | val ppl 172.27
step  3500 | lr 2.39e-04 | train loss 1.4099 | val loss 5.3297 | val ppl 206.37
step  4000 | lr 2.19e-04 | train loss 1.1406 | val loss 5.5155 | val ppl 248.50
step  4500 | lr 1.98e-04 | train loss 0.9333 | val loss 5.6914 | val ppl 296.32
step  5000 | lr 1.76e-04 | train loss 0.7750 | val loss 5.8442 | val ppl 345.21
step  5500 | lr 1.54e-04 | train loss 0.6580 | val loss 5.9720 | val 